# Домашнее задание 1

In [1]:
import csv
import json
import math
import re
from collections.abc import Callable
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

# Формат, в котором даты хранятся и выводятся в истории операций
DATE_FORMAT = '%Y-%m-%d %H:%M:%S'

## Задание 1. Базовый класс `Account`

In [2]:
class InvalidAmountError(ValueError):
    """Сумма операции некорректна (не число, ноль или отрицательная)."""


class InvalidHolderNameError(ValueError):
    """Имя владельца не соответствует формату "Имя Фамилия"."""

In [3]:
@dataclass(frozen=True)
class Operation:
    """Одна запись в истории операций счета."""

    op_type: str          # тип операции, например 'deposit'
    amount: float         # сумма операции (всегда положительная)
    date: datetime        # дата и время операции
    balance_after: float  # баланс после операции
    status: str           # 'success' или 'fail'
    comment: str = ''     # пояснение (пример: причина отказа)

    def to_dict(self) -> dict:
        """Возвращает операцию в виде словаря с датой-строкой."""
        return {
            'type': self.op_type,
            'amount': self.amount,
            'date': self.date.strftime(DATE_FORMAT),
            'balance_after': self.balance_after,
            'status': self.status,
            'comment': self.comment,
        }

    def __str__(self) -> str:
        line = (f'{self.date:{DATE_FORMAT}} | {self.op_type:<8} | '
                f'{self.amount:>10.2f} | '
                f'баланс: {self.balance_after:>10.2f} | '
                f'{self.status}')
        return f'{line} ({self.comment})' if self.comment else line

In [4]:
class Account:
    """Базовый банковский счет с историей операций и аналитикой."""

    _account_counter = 1000  # порядковый номер следующего счета
    account_type = 'base'    # тип счета
    ALLOWED_OPERATIONS = ('deposit', 'withdraw')
    STATUSES = ('success', 'fail')

    # "Имя Фамилия" с заглавных букв: только латиница или только кириллица
    NAME_PATTERN = re.compile(
        r'^(?:[A-Z][a-z]+ [A-Z][a-z]+|[А-ЯЁ][а-яё]+ [А-ЯЁ][а-яё]+)$'
    )

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        self.holder = self._validate_holder(account_holder)
        if not self._is_number(balance) or balance < 0:
            raise InvalidAmountError(
                f'Начальный баланс должен быть неотрицательным числом, '
                f'получено: {balance!r}'
            )
        # Номер присваивается только после успешной валидации всех параметров
        self.account_number = f'ACC-{Account._account_counter}'
        Account._account_counter += 1
        self._balance = round(float(balance), 2)
        self.operations_history = []
        self.rejected_records = []  # отклоненное при последней загрузке

    # Валидация
    @staticmethod
    def _is_number(value: object) -> bool:
        """True, если value - конечное число (bool числом не считается)."""
        return (isinstance(value, (int, float))
                and not isinstance(value, bool)
                and math.isfinite(value))

    @classmethod
    def _validate_holder(cls, name: object) -> str:
        """Проверяет формат имени владельца и возвращает его."""
        if not isinstance(name, str):
            raise InvalidHolderNameError('Имя владельца должно быть строкой')
        name = name.strip()
        if not cls.NAME_PATTERN.match(name):
            raise InvalidHolderNameError(
                f'Имя {name!r} должно быть в формате "Имя Фамилия" '
                f'с заглавных букв (только кириллица или только латиница)'
            )
        return name

    @classmethod
    def _validate_amount(cls, amount: object) -> float:
        """Проверяет, что сумма - положительное число; возвращает float."""
        if not cls._is_number(amount):
            raise InvalidAmountError(
                f'Сумма должна быть числом, получено: {amount!r}')
        if amount <= 0:
            raise InvalidAmountError(
                f'Сумма должна быть положительной, получено: {amount}')
        return round(float(amount), 2)

    # Операции
    def _add_operation(self, op_type: str, amount: float, status: str,
                       date: datetime | None = None,
                       comment: str = '') -> Operation:
        """Создает запись об операции и добавляет ее в историю."""
        operation = Operation(
            op_type=op_type,
            amount=amount,
            date=date or datetime.now(),
            balance_after=self._balance,
            status=status,
            comment=comment,
        )
        self.operations_history.append(operation)
        return operation

    def deposit(self, amount: float, date: datetime | None = None) -> bool:
        """Пополняет счет. Некорректная сумма вызывает InvalidAmountError."""
        amount = self._validate_amount(amount)
        self._balance = round(self._balance + amount, 2)
        self._add_operation('deposit', amount, 'success', date)
        return True

    def withdraw(self, amount: float, date: datetime | None = None) -> bool:
        """Снимает средства.

        При нехватке денег фиксирует попытку со статусом 'fail'.
        """
        amount = self._validate_amount(amount)
        if amount > self._balance:
            self._add_operation('withdraw', amount, 'fail', date,
                                'недостаточно средств')
            return False
        self._balance = round(self._balance - amount, 2)
        self._add_operation('withdraw', amount, 'success', date)
        return True

    def get_balance(self) -> float:
        """Возвращает текущий баланс."""
        return self._balance

    def get_history(self) -> list[dict]:
        """Возвращает историю операций как список словарей (копия данных)."""
        return [operation.to_dict() for operation in self.operations_history]

    def print_history(self,
                      operations: list[Operation] | None = None) -> None:
        """Печатает историю (или переданный список операций) в виде таблицы."""
        if operations is None:
            operations = self.operations_history
        print(f'История счета {self.account_number} ({self.holder}):')
        if not operations:
            print('  операций нет')
        for operation in operations:
            print(' ', operation)

    def __str__(self) -> str:
        return (f'{type(self).__name__} {self.account_number}, '
                f'владелец: {self.holder}, баланс: {self._balance:.2f}')

    def __repr__(self) -> str:
        return (f'{type(self).__name__}(account_holder={self.holder!r}, '
                f'balance={self._balance})')

    # Аналитика (задание 2, п. 4)
    def get_last_large_operations(
            self, n: int = 3,
            min_amount: float | None = None) -> list[Operation]:
        """Возвращает последние n крупных успешных операций.

        Крупной считается операция с суммой не меньше min_amount.
        Если порог не задан, берется средняя сумма успешных операций.
        Результат отсортирован от новых к старым (при равной дате -
        по убыванию суммы).
        """
        if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
            raise ValueError('n должно быть положительным целым числом')
        successful = [op for op in self.operations_history
                      if op.status == 'success']
        if not successful:
            return []
        if min_amount is None:
            min_amount = sum(op.amount for op in successful) / len(successful)
        large = [op for op in successful if op.amount >= min_amount]
        large.sort(key=lambda op: (op.date, op.amount), reverse=True)
        return large[:n]

    # Загрузка грязных данных (задание 3)
    FILE_FIELDS = ('account_number', 'account_type', 'date', 'operation',
                   'amount', 'balance_after', 'status')
    # Допустимые форматы дат: основной и сокращенный "дд/мм/гггг чч:мм"
    FILE_DATE_FORMATS = ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M')

    @staticmethod
    def _read_transactions_file(path: str | Path) -> list:
        """Читает CSV или JSON-файл и возвращает список словарей."""
        path = Path(path)
        suffix = path.suffix.lower()
        with open(path, encoding='utf-8', newline='') as file:
            if suffix == '.csv':
                return list(csv.DictReader(file))
            if suffix == '.json':
                data = json.load(file)
                if not isinstance(data, list):
                    raise ValueError(
                        'JSON-файл должен содержать список транзакций')
                return data
        raise ValueError(f'Неподдерживаемый формат файла: {suffix}')

    @staticmethod
    def _clean_text(value: object) -> str:
        """Приводит текстовое поле к нижнему регистру без пробелов по краям."""
        return str(value).strip().lower() if value is not None else ''

    @classmethod
    def _parse_date(cls, value: object) -> datetime | None:
        """Парсит дату во всех допустимых форматах."""
        for date_format in cls.FILE_DATE_FORMATS:
            try:
                return datetime.strptime(str(value).strip(), date_format)
            except ValueError:
                continue
        return None

    @staticmethod
    def _parse_amount(value: object) -> float | None:
        """Преобразует сумму в float с точностью до копеек.

        Возвращает None, если преобразование невозможно.
        """
        if (value is None or isinstance(value, bool)
                or str(value).strip() == ''):
            return None
        try:
            amount = float(value)
        except (TypeError, ValueError):
            return None
        return round(amount, 2) if math.isfinite(amount) else None

    def _parse_record(self, record: dict) -> tuple[dict, list[str]]:
        """Парсит одну запись файла и валидирует ее.

        Возвращает нормализованные поля записи и список ошибок
        (пустой, если запись валидна).
        """
        errors = []

        account_type = self._clean_text(record.get('account_type'))
        if account_type != self.account_type:
            errors.append(f'тип счета {account_type!r} не совпадает '
                          f'с {self.account_type!r}')

        operation = self._clean_text(record.get('operation'))
        if not operation:
            errors.append('не указан тип операции')
        elif operation not in self.ALLOWED_OPERATIONS:
            errors.append(f'операция {operation!r} недопустима для '
                          f'счета типа {self.account_type!r}')

        amount = self._parse_amount(record.get('amount'))
        if amount is None:
            errors.append('сумма отсутствует или не является числом')
        elif amount <= 0:
            errors.append(f'неположительная сумма {amount}')

        date = self._parse_date(record.get('date'))
        if date is None:
            errors.append(f'неверная дата {record.get("date")!r}')

        status = self._clean_text(record.get('status'))
        if status not in self.STATUSES:
            errors.append(f'неверный статус {status!r} (опечатка)')

        item = {'operation': operation, 'amount': amount,
                'date': date, 'status': status}
        return item, errors

    def clean_history(self,
                      records: list) -> tuple[list[dict], list[tuple]]:
        """Ищет ошибки в записях перед их добавлением в историю.

        Проверяются: дубликаты, тип счета, тип операции (опечатки и
        недоступные для данного счета операции), сумма (пустая, нечисловая,
        нулевая, отрицательная), дата (пустая или несуществующая) и статус.

        Возвращает кортеж (valid, rejected):
          valid    - список нормализованных записей, отсортированный по дате;
          rejected - список пар (исходная запись, список причин отказа).
        """
        valid, rejected, seen = [], [], set()

        for record in records:
            if not isinstance(record, dict):
                rejected.append((record, ['запись не является словарем']))
                continue

            item, errors = self._parse_record(record)
            key = tuple(str(record.get(field)) for field in self.FILE_FIELDS)
            if key in seen:
                errors.insert(0, 'дубликат записи')
            seen.add(key)

            if errors:
                rejected.append((record, errors))
            else:
                valid.append(item)

        valid.sort(key=lambda item: item['date'])
        return valid, rejected

    def _get_operation_handlers(self) -> dict[str, Callable]:
        """Сопоставляет тип операции из файла с методом счета."""
        return {'deposit': self.deposit, 'withdraw': self.withdraw}

    @staticmethod
    def _select_account_records(records: list, account_number: str) -> list:
        """Отбирает из общего файла записи одного счета."""
        return [
            record for record in records
            if isinstance(record, dict)
            and str(record.get('account_number', '')).strip()
            == account_number
        ]

    def _apply_records(self, items: list[dict]) -> int:
        """Применяет валидные записи к счету в хронологическом порядке.

        Возвращает число операций, отклоненных правилами счета
        (недостаток средств, лимит снятия).
        """
        handlers = self._get_operation_handlers()
        failed_by_rules = 0
        for item in items:
            if item['status'] == 'fail':
                self._add_operation(item['operation'], item['amount'], 'fail',
                                    item['date'], 'неуспешна в источнике')
                continue
            handler = handlers[item['operation']]
            if not handler(item['amount'], date=item['date']):
                failed_by_rules += 1
        return failed_by_rules

    def load_history(self, path: str | Path,
                     source_account: str | None = None) -> dict:
        """Загружает операции счета из общего CSV/JSON-файла.

        source_account - номер счета в файле (по умолчанию - номер этого
        счета). Нужен, потому что номера в общем файле могут отличаться
        от выданных программой.

        Операции применяются в хронологическом порядке и проверяются
        с учетом всех правил счета.

        Возвращает словарь со статистикой загрузки.
        """
        source_account = source_account or self.account_number
        raw_records = self._read_transactions_file(path)
        own_records = self._select_account_records(raw_records,
                                                   source_account)
        valid, rejected = self.clean_history(own_records)
        self.rejected_records = rejected
        failed_by_rules = self._apply_records(valid)

        return {
            'file': Path(path).name,
            'records_in_file': len(raw_records),
            'records_for_account': len(own_records),
            'loaded': len(valid),
            'rejected': len(rejected),
            'failed_by_rules': failed_by_rules,
            'balance': self._balance,
        }

    def print_rejected(self) -> None:
        """Печатает отклоненные при загрузке записи с причинами."""
        print(f'Отклонено записей: {len(self.rejected_records)}')
        for record, errors in self.rejected_records:
            if isinstance(record, dict):
                short = (f"{record.get('date')} | {record.get('operation')} | "
                         f"{record.get('amount')} | {record.get('status')}")
            else:
                short = repr(record)
            print(f'  {short:<50} -> {"; ".join(errors)}')

### Демонстрация задания 1

Создание счета, пополнение, снятие, попытка снять больше, чем есть на счете, и попытка внести отрицательную сумму.

In [5]:
acc = Account('Иван Петров', 1000)
print(acc)
print('Номер счета:', acc.account_number)

acc.deposit(500)
acc.withdraw(300)
print('Снятие 5000:', acc.withdraw(5000))  # нехватка средств -> fail

try:
    acc.deposit(-100)
except InvalidAmountError as error:
    print('Ошибка:', error)

print('Баланс:', acc.get_balance())
print()
acc.print_history()

Account ACC-1000, владелец: Иван Петров, баланс: 1000.00
Номер счета: ACC-1000
Снятие 5000: False
Ошибка: Сумма должна быть положительной, получено: -100
Баланс: 1200.0

История счета ACC-1000 (Иван Петров):
  2026-10-04 23:54:08 | deposit  |     500.00 | баланс:    1500.00 | success
  2026-10-04 23:54:08 | withdraw |     300.00 | баланс:    1200.00 | success
  2026-10-04 23:54:08 | withdraw |    5000.00 | баланс:    1200.00 | fail (недостаточно средств)


In [6]:
# get_history() возвращает список словарей
acc.get_history()

[{'type': 'deposit',
  'amount': 500.0,
  'date': '2026-10-04 23:54:08',
  'balance_after': 1500.0,
  'status': 'success',
  'comment': ''},
 {'type': 'withdraw',
  'amount': 300.0,
  'date': '2026-10-04 23:54:08',
  'balance_after': 1200.0,
  'status': 'success',
  'comment': ''},
 {'type': 'withdraw',
  'amount': 5000.0,
  'date': '2026-10-04 23:54:08',
  'balance_after': 1200.0,
  'status': 'fail',
  'comment': 'недостаточно средств'}]

## Задание 2

In [7]:
class CheckingAccount(Account):
    """Расчетный счет: доступны только пополнение и снятие."""

    account_type = 'checking'


class SavingsAccount(Account):
    """Сберегательный счет: проценты на остаток и лимит снятия 50 %."""

    account_type = 'savings'
    ALLOWED_OPERATIONS = Account.ALLOWED_OPERATIONS + ('interest',)
    WITHDRAW_LIMIT = 0.5  # за одну операцию можно снять не более 50 % баланса

    def withdraw(self, amount: float, date: datetime | None = None) -> bool:
        """Снимает средства, если сумма не превышает 50 % текущего баланса."""
        amount = self._validate_amount(amount)
        limit = round(self._balance * self.WITHDRAW_LIMIT, 2)
        if amount > limit:
            self._add_operation('withdraw', amount, 'fail', date,
                                f'превышен лимит снятия {limit:.2f}')
            return False

        return super().withdraw(amount, date)

    def apply_interest(self, rate: float,
                       date: datetime | None = None) -> float:
        """Начисляет проценты на остаток; rate - ставка в процентах (7 = 7 %).

        Возвращает начисленную сумму.
        """
        if not self._is_number(rate) or rate <= 0:
            raise InvalidAmountError(
                f'Ставка должна быть положительным числом, '
                f'получено: {rate!r}')
        interest = round(self._balance * rate / 100, 2)
        if interest <= 0:
            self._add_operation('interest', 0.0, 'fail', date,
                                'нулевой остаток - начислять нечего')
            return 0.0
        self._add_interest(interest, date)
        return interest

    def _add_interest(self, amount: float,
                      date: datetime | None = None) -> bool:
        """Зачисляет готовую сумму процентов."""
        amount = self._validate_amount(amount)
        self._balance = round(self._balance + amount, 2)
        self._add_operation('interest', amount, 'success', date)
        return True

    def _get_operation_handlers(self) -> dict[str, Callable]:
        """Добавляет к базовым операциям начисление процентов."""
        handlers = super()._get_operation_handlers()
        handlers['interest'] = self._add_interest
        return handlers

In [8]:
checking = CheckingAccount('Мария Иванова', 2000)
savings = SavingsAccount('John Smith', 10000)

print(checking, '| тип:', checking.account_type)
print(savings, '| тип:', savings.account_type)
print('Нумерация общая для всех типов:',
      checking.account_number, savings.account_number)

CheckingAccount ACC-1001, владелец: Мария Иванова, баланс: 2000.00 | тип: checking
SavingsAccount ACC-1002, владелец: John Smith, баланс: 10000.00 | тип: savings
Нумерация общая для всех типов: ACC-1001 ACC-1002


In [9]:
# Лимит снятия для сберегательного счета: не больше 50 % баланса
print('Снять 6000 из 10000:', savings.withdraw(6000))   # > 50 % -> fail
print('Снять 5000 из 10000:', savings.withdraw(5000))   # ровно 50 % -> success

# Проценты на остаток: 7 %
print('Начислено процентов:', savings.apply_interest(7))
print('Баланс:', savings.get_balance())
print()
savings.print_history()

Снять 6000 из 10000: False
Снять 5000 из 10000: True
Начислено процентов: 350.0
Баланс: 5350.0

История счета ACC-1002 (John Smith):
  2026-10-04 23:54:08 | withdraw |    6000.00 | баланс:   10000.00 | fail (превышен лимит снятия 5000.00)
  2026-10-04 23:54:08 | withdraw |    5000.00 | баланс:    5000.00 | success
  2026-10-04 23:54:08 | interest |     350.00 | баланс:    5350.00 | success


In [10]:
# Валидация имени владельца
names = ['Анна Смирнова', 'Peter Parker', 'анна смирнова', 'Anna',
         'Ivan Петров', 'ANNA SMITH', 'Anna  Smith', 'Anna Smith2', 42]

for name in names:
    try:
        CheckingAccount(name)
        print(f'{name!r:<18} -> OK')
    except InvalidHolderNameError as error:
        print(f'{name!r:<18} -> {error}')

'Анна Смирнова'    -> OK
'Peter Parker'     -> OK
'анна смирнова'    -> Имя 'анна смирнова' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
'Anna'             -> Имя 'Anna' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
'Ivan Петров'      -> Имя 'Ivan Петров' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
'ANNA SMITH'       -> Имя 'ANNA SMITH' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
'Anna  Smith'      -> Имя 'Anna  Smith' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
'Anna Smith2'      -> Имя 'Anna Smith2' должно быть в формате "Имя Фамилия" с заглавных букв (только кириллица или только латиница)
42                 -> Имя владельца должно быть строкой


In [11]:
# Валидация сумм
for bad_amount in [-100, 0, 'сто', None, float('nan'), True]:
    try:
        checking.deposit(bad_amount)
    except InvalidAmountError as error:
        print(f'deposit({bad_amount!r}) -> {error}')

try:
    CheckingAccount('Олег Сидоров', -50)
except InvalidAmountError as error:
    print('Отрицательный начальный баланс ->', error)

try:
    savings.apply_interest(-5)
except InvalidAmountError as error:
    print('Отрицательная ставка ->', error)

deposit(-100) -> Сумма должна быть положительной, получено: -100
deposit(0) -> Сумма должна быть положительной, получено: 0
deposit('сто') -> Сумма должна быть числом, получено: 'сто'
deposit(None) -> Сумма должна быть числом, получено: None
deposit(nan) -> Сумма должна быть числом, получено: nan
deposit(True) -> Сумма должна быть числом, получено: True
Отрицательный начальный баланс -> Начальный баланс должен быть неотрицательным числом, получено: -50
Отрицательная ставка -> Ставка должна быть положительным числом, получено: -5


In [12]:
# Анализ истории: последние n крупных операций

analytics = CheckingAccount('Elena Volkova', 0)
start = datetime(2025, 10, 1, 12, 0)
for day, (method, amount) in enumerate([
        (analytics.deposit, 5000), (analytics.withdraw, 200),
        (analytics.deposit, 1500), (analytics.withdraw, 3000),
        (analytics.deposit, 100), (analytics.deposit, 2500),
        (analytics.withdraw, 50), (analytics.withdraw, 4000)]):
    method(amount, date=start + timedelta(days=day))

analytics.print_history()

print('\nПоследние 3 крупные операции (порог = средняя сумма):')
for operation in analytics.get_last_large_operations(3):
    print(' ', operation)

print('\nПоследние 2 операции на сумму от 2000:')
for operation in analytics.get_last_large_operations(2, min_amount=2000):
    print(' ', operation)

История счета ACC-1005 (Elena Volkova):
  2025-10-01 12:00:00 | deposit  |    5000.00 | баланс:    5000.00 | success
  2025-10-02 12:00:00 | withdraw |     200.00 | баланс:    4800.00 | success
  2025-10-03 12:00:00 | deposit  |    1500.00 | баланс:    6300.00 | success
  2025-10-04 12:00:00 | withdraw |    3000.00 | баланс:    3300.00 | success
  2025-10-05 12:00:00 | deposit  |     100.00 | баланс:    3400.00 | success
  2025-10-06 12:00:00 | deposit  |    2500.00 | баланс:    5900.00 | success
  2025-10-07 12:00:00 | withdraw |      50.00 | баланс:    5850.00 | success
  2025-10-08 12:00:00 | withdraw |    4000.00 | баланс:    1850.00 | success

Последние 3 крупные операции (порог = средняя сумма):
  2025-10-08 12:00:00 | withdraw |    4000.00 | баланс:    1850.00 | success
  2025-10-06 12:00:00 | deposit  |    2500.00 | баланс:    5900.00 | success
  2025-10-04 12:00:00 | withdraw |    3000.00 | баланс:    3300.00 | success

Последние 2 операции на сумму от 2000:
  2025-10-08 12:00

## Задание 3. Загрузка грязных данных

! Файлы `transactions_dirty.csv` и `transactions_dirty.json` должны лежать рядом с ноутбуком.

In [13]:
DATA_FILES = ['transactions_dirty.csv', 'transactions_dirty.json']
missing = [name for name in DATA_FILES if not Path(name).exists()]

if missing:
    raise FileNotFoundError(
        'Положите файлы рядом с ноутбуком: ' + ', '.join(missing))
print('Файлы с данными найдены:', ', '.join(DATA_FILES))

Файлы с данными найдены: transactions_dirty.csv, transactions_dirty.json


In [14]:
# Расчетный счет: загрузка операций счета ACC-100001 из CSV
checking_loaded = CheckingAccount('Иван Петров', 1200)
report = checking_loaded.load_history('transactions_dirty.csv',
                                      source_account='ACC-100001')
print(report)
print()
checking_loaded.print_rejected()
print()
checking_loaded.print_history()

{'file': 'transactions_dirty.csv', 'records_in_file': 105, 'records_for_account': 26, 'loaded': 17, 'rejected': 9, 'failed_by_rules': 0, 'balance': 4864.0}

Отклонено записей: 9
  2025-09-29 22:17:26 | withdraw |  | success        -> сумма отсутствует или не является числом
  2025-10-01 22:17:26 |  | 654.0 | success           -> не указан тип операции
  2025-10-03 22:17:26 | interest | 532.0 | success   -> операция 'interest' недопустима для счета типа 'checking'
  2025-17-34 12:00:00 | deposit | 793.0 | success    -> неверная дата '2025-17-34 12:00:00'
  2025-10-06 22:17:26 | diposit | 669.0 | success    -> операция 'diposit' недопустима для счета типа 'checking'
  2025-10-08 22:17:26 | interest | 744.0 | success   -> операция 'interest' недопустима для счета типа 'checking'
  2023-16-40 12:00:00 | withdraw | 111.0 | success   -> неверная дата '2023-16-40 12:00:00'
  2025-10-19 22:17:26 | deposit | 916.0 | succes     -> неверный статус 'succes' (опечатка)
  2025-10-16 22:17:26 | withd

In [15]:
# Сберегательный счет: загрузка операций счета ACC-100002 из JSON
savings_loaded = SavingsAccount('Anna Smirnova', 1500)
report = savings_loaded.load_history('transactions_dirty.json',
                                     source_account='ACC-100002')
print(report)
print()
savings_loaded.print_rejected()
print()
savings_loaded.print_history()

{'file': 'transactions_dirty.json', 'records_in_file': 105, 'records_for_account': 27, 'loaded': 18, 'rejected': 9, 'failed_by_rules': 0, 'balance': 3991.24}

Отклонено записей: 9
  2023-22-32 12:00:00 | deposit | 797.0 | success    -> неверная дата '2023-22-32 12:00:00'
  2025-09-30 22:17:26 | interest | None | success    -> сумма отсутствует или не является числом
  2025-10-08 22:17:26 | deposit | -136.0 | success   -> неположительная сумма -136.0
  2025-10-09 22:17:26 | diposit | 432.0 | success    -> операция 'diposit' недопустима для счета типа 'savings'
  2025-10-16 22:17:26 | deposit | -872.0 | success   -> неположительная сумма -872.0
  2025-10-19 22:17:26 | withdraw | 300.0 | succes    -> неверный статус 'succes' (опечатка)
  2025-10-22 22:17:26 | wdraw | 457.0 | success      -> операция 'wdraw' недопустима для счета типа 'savings'
  09/10/2025 22:17 | deposit | 638.0 | success       -> дубликат записи
  2025-10-19 22:17:26 | withdraw | 300.0 | succes    -> дубликат записи; не

In [16]:
# Сводка по всем счетам из обоих файлов: CSV и JSON дают одинаковый результат
source_accounts = [
    ('ACC-100001', CheckingAccount, 'Иван Петров', 1200),
    ('ACC-100002', SavingsAccount, 'Anna Smirnova', 1500),
    ('ACC-100003', CheckingAccount, 'Ольга Белова', 800),
    ('ACC-100004', SavingsAccount, 'Mark Brown', 2000),
]

header = (f"{'Счет в файле':<13}{'Класс':<16}{'Файл':<25}{'Строк':>6}"
          f"{'Принято':>9}{'Отклонено':>11}{'Баланс':>11}")
print(header)
print('-' * len(header))
for number, account_class, holder, start_balance in source_accounts:
    for file_name in DATA_FILES:
        account = account_class(holder, start_balance)
        result = account.load_history(file_name, source_account=number)
        print(f"{number:<13}{account_class.__name__:<16}{file_name:<25}"
              f"{result['records_for_account']:>6}{result['loaded']:>9}"
              f"{result['rejected']:>11}{result['balance']:>11.2f}")

Счет в файле Класс           Файл                      Строк  Принято  Отклонено     Баланс
-------------------------------------------------------------------------------------------
ACC-100001   CheckingAccount transactions_dirty.csv       26       17          9    4864.00
ACC-100001   CheckingAccount transactions_dirty.json      26       17          9    4864.00
ACC-100002   SavingsAccount  transactions_dirty.csv       27       18          9    3991.24
ACC-100002   SavingsAccount  transactions_dirty.json      27       18          9    3991.24
ACC-100003   CheckingAccount transactions_dirty.csv       21       13          8    5446.00
ACC-100003   CheckingAccount transactions_dirty.json      21       13          8    5446.00
ACC-100004   SavingsAccount  transactions_dirty.csv       31       19         12   10057.25
ACC-100004   SavingsAccount  transactions_dirty.json      31       19         12   10057.25


In [17]:
# Фильтрация по типу счета: записи расчетного счета не попадут в сберегательный
wrong_type = SavingsAccount('Петр Козлов', 1000)
print(wrong_type.load_history('transactions_dirty.csv',
                              source_account='ACC-100001'))
print('История пуста:', wrong_type.get_history() == [])

{'file': 'transactions_dirty.csv', 'records_in_file': 105, 'records_for_account': 26, 'loaded': 0, 'rejected': 26, 'failed_by_rules': 0, 'balance': 1000.0}
История пуста: True


In [18]:
# Проверка аналитики на загруженной истории
print('Последние 5 крупных операций сберегательного счета:')
for operation in savings_loaded.get_last_large_operations(5):
    print(' ', operation)

Последние 5 крупных операций сберегательного счета:
  2025-10-13 22:17:26 | withdraw |     440.00 | баланс:    4213.98 | success
  2025-10-09 22:17:26 | deposit  |     622.00 | баланс:    4625.72 | success
  2025-10-09 22:17:00 | deposit  |     638.00 | баланс:    4173.72 | success
  2025-10-08 22:17:26 | deposit  |     426.00 | баланс:    3535.72 | success
  2025-10-02 22:17:26 | deposit  |     840.00 | баланс:    2734.00 | success
